<div style="text-align: center;">
<a target="_blank" href="https://colab.research.google.com/github/miquelmn/aa_2627/blob/main/07_Word2Vec/Word2Vec_PyTorch.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
</div>

In [ ]:
import gzip
import re
import time
import urllib.request
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import (
    BatchSampler,
    DataLoader,
    Dataset,
    RandomSampler,
    TensorDataset,
)


# Representació de paraules i *word2vec* amb PyTorch

En aquesta pràctica canviarem de domini: fins ara hem treballat amb imatges, però ara ens centrarem en el **llenguatge natural** (*Natural Language Processing*, NLP). El primer problema que trobam és que les xarxes neuronals només entenen números, així que hem de decidir com representam una paraula. Veurem que la manera de fer-ho condiciona tot el que ve després, i implementarem i entrenarem **word2vec** amb **PyTorch**.

### word2vec

**word2vec** és una família de models proposada per Tomas Mikolov i el seu equip a Google l'any 2013 que aprèn una representació vectorial densa de cada paraula a partir de grans quantitats de text sense etiquetar. Els dos articles de referència són:
- [Efficient Estimation of Word Representations in Vector Space, 2013](https://arxiv.org/abs/1301.3781).
- [Distributed Representations of Words and Phrases and their Compositionality, 2013](https://arxiv.org/abs/1310.4546).

#### 1. Context i motivació

La solució més directa per representar una paraula és la codificació ***one-hot***: un vector de la mida del vocabulari ($|V|$) amb un 1 a la posició de la paraula i 0 a la resta. Aquesta representació té diversos problemes:
* **Dimensionalitat:** un vocabulari de 100.000 paraules genera vectors de 100.000 dimensions, quasi tot zeros.
* **Cap noció de semblança:** tots els vectors són ortogonals entre ells, així que *gat* és tan diferent de *gos* com de *democràcia*.

La clau de word2vec és la **hipòtesi distribucional**:

> "You shall know a word by the company it keeps." (J. R. Firth, 1957)

És a dir, les paraules que apareixen en contextos semblants tenen significats semblants. Això ens permet un nou paradigma:
1. **Aprenentatge autosupervisat:** no necessitam etiquetes, el propi text ens dona les parelles (paraula, context).
2. **Vectors densos de poques dimensions:** normalment entre 50 i 300 números reals per paraula, anomenats ***embeddings***.
3. **Aritmètica semàntica:** les relacions entre paraules es codifiquen com a direccions de l'espai (rei $-$ home $+$ dona $\approx$ reina).

#### 2. Arquitectures

Existeixen dues variants de word2vec que es diferencien en què es prediu i a partir de què:

| Model | Entrada | Sortida | Característiques |
| :--- | :--- | :--- | :--- |
| **Skip-gram** | Paraula central | Paraules del context | Més lent, funciona millor amb paraules poc freqüents |
| **CBOW** (*Continuous Bag of Words*) | Paraules del context | Paraula central | Més ràpid, funciona millor amb paraules freqüents |

En tots dos casos es defineix una **finestra de context** de mida $m$: per a cada paraula central considerarem les $m$ paraules anteriors i les $m$ posteriors. En aquesta pràctica implementarem el *skip-gram*.

La xarxa té una única capa oculta lineal, sense funció d'activació:

$$\text{one-hot}(c) \;\longrightarrow\; W_{in} \in \mathbb{R}^{|V| \times D} \;\longrightarrow\; h = v_c \;\longrightarrow\; W_{out} \in \mathbb{R}^{D \times |V|} \;\longrightarrow\; \text{softmax}$$

Després de l'entrenament, les files de $W_{in}$ són els *embeddings* que cercam.

#### 3. Funció de pèrdua

Per al *skip-gram*, la probabilitat de veure la paraula de context $o$ donada la central $c$ es defineix amb un *softmax*:

$$P(o \mid c) = \frac{\exp(u_o^\top v_c)}{\sum_{w \in V} \exp(u_w^\top v_c)}$$

i la pèrdua d'una parella és l'*entropia creuada* $L = -\log P(o \mid c)$. El problema és que el denominador suma sobre **tot** el vocabulari, i això és molt car. Per això a la pràctica s'empra ***negative sampling***: en lloc de predir entre totes les paraules, el model ha de distingir el context real de $K$ paraules triades a l'atzar (els *negatius* $w_k$):

$$L = -\log \sigma(u_o^\top v_c) - \sum_{k=1}^{K} \log \sigma(-u_{w_k}^\top v_c)$$

Els negatius no es trien de manera uniforme, sinó segons la distribució de freqüències elevada a $3/4$, $P_n(w) \propto f(w)^{3/4}$, que dona una mica més de pes a les paraules rares. A més, word2vec descarta aleatòriament les paraules molt freqüents (*subsampling*, com *el* o *de*), ja que aporten poca informació. Una paraula amb freqüència $f(w)$ es manté amb probabilitat $\min\left(1, \sqrt{t / f(w)}\right)$, on $t$ és un llindar (típicament entre $10^{-5}$ i $10^{-3}$).

### Similitud i analogies

Com que cada paraula és un vector, podem mesurar la seva semblança amb la **similitud cosinus**:

$$\cos(u, v) = \frac{u \cdot v}{\|u\| \, \|v\|}$$

Les **analogies** es resolen amb aritmètica de vectors: per respondre «*home* és a *rei* com *dona* és a ...?» cercam la paraula més propera a $v_{rei} - v_{home} + v_{dona}$.

### Word2vec i PyTorch: `nn.Embedding`

Multiplicar un vector *one-hot* per una matriu $W_{in}$ és equivalent a **seleccionar una fila** d'aquesta matriu. PyTorch ho implementa de manera eficient amb la capa [`nn.Embedding`](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html): una taula de cerca de mida $|V| \times D$ que, donat un índex (o un lot d'índexs), retorna els vectors corresponents i que s'entrena per retropropagació com qualsevol altra capa. Per tant, el nostre model necessitarà dues capes `nn.Embedding`: una per a les paraules centrals ($W_{in}$) i una per a les de context ($W_{out}$).

## Començam

Primer de tot farem una implementació amb el *softmax* complet sobre un corpus minúscul, per entendre què passa dins del model. Després implementarem el ***negative sampling*** i entrenarem amb un corpus real.

### Què fa `nn.Embedding`?

Comprovem amb un exemple petit que una capa `nn.Embedding` és el mateix que multiplicar per un vector *one-hot*:

In [ ]:
emb = nn.Embedding(num_embeddings=5, embedding_dim=3)
idx = torch.tensor([1, 3])

print(emb(idx))                                   # fa una cerca per índex

one_hot = F.one_hot(idx, num_classes=5).float()   # equivalent a multiplicar per one-hot
print(torch.allclose(one_hot @ emb.weight, emb(idx)))

### Skip-gram amb *softmax* (corpus petit)

Per poder veure bé els resultats hem dissenyat un corpus petit en català amb estructura semàntica clara (reis i reines, homes i dones, animals...). Primer de tot l'hem de tokenitzar i construir el vocabulari.

In [ ]:
corpus_text = """
el rei governa el regne gran
la reina governa el regne gran
el rei estima la reina
la reina estima el rei
un home treballa al camp
una dona treballa al camp
un home estima una dona
una dona estima un home
el gat beu llet
el gos beu aigua
el gat menja peix
el gos menja carn
el gat dorm al sofà
el gos dorm al jardí
el nen juga al jardí
la nena juga al jardí
el nen estima el gos
la nena estima el gat
el rei i la reina viuen al palau
un home i una dona viuen a la casa
el nen i la nena viuen a la casa
"""

def tokenize(line):
    return re.findall(r"[a-zàèéíòóúïüç]+", line.lower())

sentences_toy = [tokenize(l) for l in corpus_text.strip().split("\n")]

vocab = sorted({w for s in sentences_toy for w in s})
word2id = {w: i for i, w in enumerate(vocab)}
sentences_ids = [[word2id[w] for w in s] for s in sentences_toy]

print(f"{len(sentences_toy)} frases, vocabulari de {len(vocab)} paraules")
print(vocab)

Ara hem de generar les parelles d'entrenament **(paraula central, paraula de context)**. Per a cada posició `i` de la frase agafam totes les paraules a distància com a màxim `window`, sense comptar la mateixa paraula. Amb les parelles construïm un `TensorDataset` i un `DataLoader`, exactament com vàrem fer a les sessions anteriors.

In [ ]:
def generate_pairs(ids, window=2):
    pairs = []
    for i, c in enumerate(ids):
        for j in range(max(0, i - window), min(len(ids), i + window + 1)):
            if j != i:
                pairs.append((c, ids[j]))
    return pairs

all_pairs = [p for s in sentences_ids for p in generate_pairs(s, window=2)]
print("Total de parelles d'entrenament:", len(all_pairs))
print([(vocab[c], vocab[o]) for c, o in all_pairs[:6]])

centers = torch.tensor([c for c, _ in all_pairs])
contexts = torch.tensor([o for _, o in all_pairs])

toy_loader = DataLoader(TensorDataset(centers, contexts), batch_size=32, shuffle=True)

c_batch, o_batch = next(iter(toy_loader))
print(c_batch.shape, o_batch.shape)

Definim ara el model. És un `nn.Module` amb una capa `nn.Embedding` per a $W_{in}$ i una capa lineal per a $W_{out}$, que retorna els ***logits*** de les $|V|$ paraules possibles. Com que `nn.CrossEntropyLoss` ja aplica el *softmax* internament, la pèrdua serà exactament $-\log P(o \mid c)$.

In [ ]:
class SkipGramSoftmax(nn.Module):
    def __init__(self, vocab_size, dim=10):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, dim)
        self.out = nn.Linear(dim, vocab_size, bias=False)

    def forward(self, center):
        h = self.in_embed(center)    # (B, D)
        return self.out(h)           # (B, V) logits


model_toy = SkipGramSoftmax(len(vocab), dim=10)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_toy.parameters(), lr=0.01)

Per explorar els resultats definim una funció que, donats uns *embeddings* **normalitzats**, retorna les paraules més properes segons la similitud cosinus. Accepta una llista de paraules `positive` que se sumen i una de `negative` que es resten, de manera que també serveix per resoldre analogies. Els *embeddings* d'un model són les files de la matriu `in_embed.weight`.

In [ ]:
def get_embeddings(model):
    """Embeddings normalitzats (norma 1) de la capa d'entrada."""
    return F.normalize(model.in_embed.weight.detach().cpu(), dim=1)


def most_similar(emb, word2id, vocab, positive, negative=(), topn=5):
    pos = [word2id[w] for w in positive]
    neg = [word2id[w] for w in negative]

    query = emb[pos].sum(dim=0) - (emb[neg].sum(dim=0) if neg else 0)
    query = F.normalize(query, dim=0)

    sims = emb @ query                       # similitud cosinus amb totes les paraules
    sims[pos + neg] = -float("inf")          # excloem les paraules de la consulta
    top = sims.topk(topn)
    return [(vocab[i], round(s.item(), 3)) for s, i in zip(top.values, top.indices)]

Una vegada entrenat `model_toy`, cridar `most_similar(get_embeddings(model_toy), word2id, vocab, positive=["rei"])` ens retornaria les paraules més properes a *rei*. **Aquest entrenament el heu de fer vosaltres a la feina a fer**; ara passam a un corpus real.

### word2vec amb *negative sampling* (corpus real)

El nostre model és útil per entendre els conceptes, però amb 21 frases no n'aprendrà res d'interessant. A més, el *softmax* complet és inviable amb vocabularis grans. El corpus que emprarem ara és **text8**, els primers 100 MB de la Viquipèdia anglesa neta (unes 17 milions de paraules). És un únic fitxer de text amb totes les paraules separades per espais.

Podem baixar-lo amb Python o, alternativament, amb l'eina ``wget`` així:

```
wget https://github.com/RaRe-Technologies/gensim-data/releases/download/text8/text8.gz
```

In [ ]:
URL = "https://github.com/RaRe-Technologies/gensim-data/releases/download/text8/text8.gz"
urllib.request.urlretrieve(URL, "text8.gz")

with gzip.open("text8.gz", "rt") as f:
    tokens = f.read().split()

print("Paraules totals:", len(tokens))
print(tokens[:15])

#### Preprocessament

Aquest cop el corpus és molt gran i hem de preparar les dades amb cura perquè l'entrenament sigui ràpid. Els passos són:
1. **Construir el vocabulari**, descartant les paraules amb menys de `MIN_COUNT` aparicions.
2. **Fer *subsampling*** de les paraules molt freqüents.
3. **Generar les parelles** (central, context) de manera vectoritzada.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

MAX_TOKENS = 5_000_000   
MIN_COUNT = 5
SUBSAMPLE_T = 1e-4
WINDOW = 5
EMBED_DIM = 100
NEGATIVES = 5
BATCH_SIZE = 4096
EPOCHS = 5
LR = 3e-3

tokens = tokens[:MAX_TOKENS]
counter = Counter(tokens)

text8_vocab = [w for w, c in counter.most_common() if c >= MIN_COUNT]
text8_word2id = {w: i for i, w in enumerate(text8_vocab)}
counts = np.array([counter[w] for w in text8_vocab], dtype=np.float64)

ids = np.array([text8_word2id[w] for w in tokens if w in text8_word2id], dtype=np.int64)
print(f"Vocabulari: {len(text8_vocab)} paraules · {len(ids)} tokens")

# Subsampling: es manté cada token amb probabilitat min(1, sqrt(t / f(w)))
freq = counts / counts.sum()
keep_prob = np.minimum(1.0, np.sqrt(SUBSAMPLE_T / freq))
ids = ids[np.random.rand(len(ids)) < keep_prob[ids]]
print(f"Després del subsampling: {len(ids)} tokens")

Per generar les parelles, per a cada desplaçament $\pm 1, \ldots, \pm m$ aparellam la seqüència amb ella mateixa desplaçada. Així evitam un bucle de Python sobre milions de paraules.

In [ ]:
def build_pairs(ids, window):
    cs, os_ = [], []
    for off in range(1, window + 1):
        cs += [ids[:-off], ids[off:]]
        os_ += [ids[off:], ids[:-off]]
    return torch.from_numpy(np.concatenate(cs)), torch.from_numpy(np.concatenate(os_))


centers_big, contexts_big = build_pairs(ids, WINDOW)
print("Parelles d'entrenament:", len(centers_big))

Amb aquesta quantitat de parelles, un `DataLoader` estàndard (que crida `__getitem__` una vegada per cada parella) seria massa lent. Ho podem resoldre fent que el `Dataset` retorni **un lot sencer** a cada crida: li passam un `BatchSampler` com a *sampler* i desactivam el *batching* automàtic amb `batch_size=None`.

In [ ]:
class PairsDataset(Dataset):
    def __init__(self, centers, contexts):
        self.centers = centers
        self.contexts = contexts

    def __len__(self):
        return len(self.centers)

    def __getitem__(self, idx):          # idx és una llista d'índexs (un lot)
        return self.centers[idx], self.contexts[idx]


pairs_ds = PairsDataset(centers_big, contexts_big)
big_loader = DataLoader(
    pairs_ds,
    sampler=BatchSampler(RandomSampler(pairs_ds), batch_size=BATCH_SIZE, drop_last=False),
    batch_size=None,
)

c_batch, o_batch = next(iter(big_loader))
print(c_batch.shape, o_batch.shape)

#### El model

El model amb *negative sampling* rep, per a cada parella, la paraula central, la de context i els $K$ negatius, i retorna directament la pèrdua $L$ promitjada sobre el lot:

$$L = -\log \sigma(u_o^\top v_c) - \sum_{k=1}^{K} \log \sigma(-u_{w_k}^\top v_c)$$

`F.logsigmoid` calcula $\log \sigma(x)$ de manera numèricament estable. Inicialitzam $W_{in}$ amb valors petits i $W_{out}$ amb zeros, com fa la implementació original.

In [ ]:
class SkipGramNS(nn.Module):
    def __init__(self, vocab_size, dim=100):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, dim)
        self.out_embed = nn.Embedding(vocab_size, dim)
        nn.init.uniform_(self.in_embed.weight, -0.5 / dim, 0.5 / dim)
        nn.init.zeros_(self.out_embed.weight)

    def forward(self, center, context, negatives):
        v = self.in_embed(center)                       # (B, D)
        u_pos = self.out_embed(context)                 # (B, D)
        u_neg = self.out_embed(negatives)               # (B, K, D)

        pos_score = (v * u_pos).sum(dim=1)              # (B,)
        neg_score = torch.bmm(u_neg, v.unsqueeze(2)).squeeze(2)   # (B, K)

        loss = -(F.logsigmoid(pos_score) + F.logsigmoid(-neg_score).sum(dim=1))
        return loss.mean()

#### Entrenament

Els negatius s'han de mostrejar segons $P_n(w) \propto f(w)^{3/4}$, que calculam una vegada i usam amb `torch.multinomial` a cada lot.

Com vàrem veure a la sessió anterior, **entrenar a CPU és molt lent** i convé emprar una **GPU**. Aquí també hem de moure el model i les dades al mateix *device*. Si empram Google Colab, hem d'activar-la a *Entorn d'execució → Canvia el tipus d'entorn d'execució → GPU*.

In [ ]:
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

noise_dist = torch.tensor(counts ** 0.75, dtype=torch.float)
noise_dist = (noise_dist / noise_dist.sum()).to(device)

model_w2v = SkipGramNS(len(text8_vocab), dim=EMBED_DIM)

model_w2v.to(device)
optimizer = optim.Adam(model_w2v.parameters(), lr=LR)
history = []

pbar = tqdm(range(EPOCHS))
for epoch in pbar:
    start = time.time()
    total, n_batches = 0.0, 0
    for centers, contexts in big_loader:
        centers, contexts = centers.to(device), contexts.to(device)
        negatives = torch.multinomial(noise_dist, centers.size(0) * NEGATIVES, replacement=True)
        negatives = negatives.view(-1, NEGATIVES)

        loss = model_w2v(centers, contexts, negatives)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()
        n_batches += 1
    history.append(total / n_batches)
    pbar.set_description(f"Època {epoch + 1}/{EPOCHS} · pèrdua {history[-1]:.4f} · {time.time() - start:.0f} s")

plt.plot(range(1, len(history) + 1), history, marker="o")
plt.xlabel("època")
plt.ylabel("pèrdua mitjana")
plt.title("Entrenament skip-gram amb negative sampling")
plt.show()

#### Resultats

Ara ja podem emprar els *embeddings* apresos. Comencem pels veïns més propers d'algunes paraules:

In [ ]:
emb = get_embeddings(model_w2v)

for w in ["king", "france", "dog", "computer"]:
    if w in text8_word2id:
        print(w, "→", most_similar(emb, text8_word2id, text8_vocab, positive=[w], topn=6))

Ara provam l'analogia clàssica: $v_{king} - v_{man} + v_{woman} \approx v_{queen}$. Les paraules que sumam van a `positive` i les que restam a `negative`. Amb un corpus tan petit com aquest no sempre funciona, perquè els models professionals s'entrenen amb milers de milions de paraules.

In [ ]:
print(most_similar(emb, text8_word2id, text8_vocab, positive=["king", "woman"], negative=["man"]))
print(most_similar(emb, text8_word2id, text8_vocab, positive=["paris", "italy"], negative=["france"]))

### Visualització dels *embeddings*

Els vectors tenen 100 dimensions, així que no els podem dibuixar directament. Emprarem **PCA** per projectar-los a 2D i comprovar si les paraules de la mateixa categoria queden agrupades.

In [ ]:
from sklearn.decomposition import PCA

groups = {
    "països":   ["france", "germany", "italy", "spain", "russia", "china", "japan"],
    "capitals": ["paris", "berlin", "rome", "madrid", "moscow", "beijing", "tokyo"],
    "animals":  ["dog", "cat", "horse", "cow", "bird", "fish", "lion"],
    "nombres":  ["one", "two", "three", "four", "five", "six", "seven"],
}
colors = ["tab:red", "tab:blue", "tab:green", "tab:orange"]

pca = PCA(n_components=2).fit(emb.numpy())   # ajustat amb tot el vocabulari

plt.figure(figsize=(9, 7))
for (name, ws), col in zip(groups.items(), colors):
    ws = [w for w in ws if w in text8_word2id]
    pts = pca.transform(emb[[text8_word2id[w] for w in ws]].numpy())
    plt.scatter(pts[:, 0], pts[:, 1], c=col, label=name)
    for (x, y), w in zip(pts, ws):
        plt.annotate(w, (x, y), fontsize=9, xytext=(3, 3), textcoords="offset points")
plt.legend()
plt.title("Embeddings word2vec (PCA)")
plt.show()

## EXTRA: Models preentrenats

Igual que a la sessió anterior, **normalment no entrenarem els nostres *embeddings* des de zero**, sinó que emprarem models ja preentrenats amb corpus enormes. La llibreria `gensim` ens en permet descarregar diversos:

| Model | Corpus | Mida |
| :--- | :--- | :--- |
| `glove-wiki-gigaword-100` | Viquipèdia + Gigaword (GloVe, 100 dimensions) | ~130 MB |
| `word2vec-google-news-300` | Google News (word2vec original, 300 dimensions) | ~1,6 GB |

Només l'emprarem per baixar els vectors: després els convertim a tensors de PyTorch i emprarem la capa [`nn.Embedding.from_pretrained`](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html#torch.nn.Embedding.from_pretrained), que és la manera habitual d'inicialitzar la primera capa d'una xarxa per a NLP (per exemple, un classificador de text) amb *embeddings* ja entrenats. Podeu veure tots els models disponibles amb `api.info()["models"].keys()`.

In [ ]:


GLOVE_URL = "https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-wiki-gigaword-100/glove-wiki-gigaword-100.gz"
MAX_WORDS = 100_000


def load_glove(url, filename, max_words):
    urllib.request.urlretrieve(url, filename)
    words, vectors = [], []
    with gzip.open(filename, "rt", encoding="utf8") as f:
        next(f)                                   # capçalera: nombre de paraules i dimensió
        for line in f:
            parts = line.rstrip().split(" ")
            words.append(parts[0])
            vectors.append(np.asarray(parts[1:], dtype=np.float32))
            if len(words) == max_words:
                break
    return words, torch.from_numpy(np.stack(vectors))


glove_vocab, glove_vectors = load_glove(GLOVE_URL, "glove-100.gz", MAX_WORDS)
glove_word2id = {w: i for i, w in enumerate(glove_vocab)}

glove_layer = nn.Embedding.from_pretrained(glove_vectors, freeze=True)
glove_emb = F.normalize(glove_layer.weight, dim=1)

print(glove_layer)
print(most_similar(glove_emb, glove_word2id, glove_vocab, positive=["king", "woman"], negative=["man"], topn=3))
print(most_similar(glove_emb, glove_word2id, glove_vocab, positive=["barcelona"], topn=5))

Els *embeddings* aprenen de textos escrits per persones i, per tant, **hereten els seus biaixos**. Fixau-vos en el que passa amb les professions:

In [ ]:
for target in ["doctor", "programmer", "nurse"]:
    res = most_similar(glove_emb, glove_word2id, glove_vocab, positive=[target, "woman"], negative=["man"], topn=3)
    print(target, "- man + woman →", [w for w, _ in res])

## Feina a fer:

1. Skip-gram amb *softmax* (corpus petit):
   1. Escriure el bucle d'entrenament per a `model_toy` emprant `toy_loader`, `criterion` i `optimizer`: guardar la **pèrdua mitjana** de cada època i dibuixar la corba de pèrdua.
   2. Comprovar els veïns més propers de `rei`, `moix`, `home` i `juga` amb `most_similar(get_embeddings(model_toy), word2id, vocab, positive=[...])`. Tenen sentit? Què passa si canviau `dim`, `lr`, el nombre d'èpoques o la finestra?
   3. Entrenar el mateix corpus amb `SkipGramNS` (haureu de mostrejar els negatius amb `torch.multinomial` o `torch.randint`) i comparar la pèrdua i els resultats amb els del *softmax* complet.
2. word2vec amb *negative sampling* (text8):
   1. Implementar una funció `analogy_accuracy(emb, word2id, vocab, analogies)` que calculi l'exactitud *top-1* sobre el conjunt d'analogies de Google (vegeu la pista). Quina exactitud obteniu?
   2. Avaluar l'efecte dels hiperparàmetres (`EMBED_DIM`, `WINDOW`, `NEGATIVES` i `SUBSAMPLE_T`) sobre aquesta exactitud. Per anar més ràpid, emprau un `MAX_TOKENS` més petit i poques èpoques. Presentau els resultats en una taula o un gràfic.
   3. Visualitzar els mateixos grups de paraules amb **t-SNE** (`sklearn.manifold.TSNE`) i afegir-hi un grup propi. Quines diferències hi ha respecte a PCA?
   4. Implementar el model **CBOW**: per a cada paraula central, fer la mitjana dels *embeddings* de les $2m$ paraules del context (`emb.mean(dim=1)`) i predir la central. Comparar-lo amb *skip-gram*.
3. Models preentrenats:
   1. Calcular l'exactitud sobre les mateixes analogies amb GloVe i comparar-la amb la del vostre model. Quin és millor? Per què?
   2. Explorar biaixos amb altres parelles de paraules. Quins riscs té emprar aquests *embeddings* en un sistema real?

**Nota**. Com veureu no us donam aquesta vegada el bucle d'entrenament del model amb *softmax*, sigui com sigui podeu adaptar el vist a les sessions anteriors.

**Pista** per a la tasca 2.1. El conjunt d'analogies de Google té línies de la forma `a b c d`, que es llegeixen «*a* és a *b* com *c* és a *d*». Per tant, hem de predir $d$ a partir de $v_b - v_a + v_c$. Les línies que comencen per `:` són capçaleres de secció i les analogies amb paraules fora del vostre vocabulari s'han d'ignorar.



In [ ]:
url = "https://raw.githubusercontent.com/nicholas-leonard/word2vec/master/questions-words.txt"
urllib.request.urlretrieve(url, "questions-words.txt")

analogies = [l.lower().split() for l in open("questions-words.txt") if not l.startswith(":")]
analogies
